# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
import json
!git clone https://github.com/Vedika1304-05/flyrank-internship-ml.git
%cd flyrank-internship-ml

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 170 (delta 74), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 1.93 MiB | 2.29 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/flyrank-internship-ml/flyrank-internship-ml


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The ML model which we are trying to build here is a scoring based model whose output produces a ranking, which is used to identify the top 50 pages that the content team must refresh first.
We aren't doing any type of clutering here. We're just assigning a score (number) to each of the content items (pages) in our dataset and then ranking all these pages using the score given to them. We are using the features & observations that we have for a particular page & trying to assign that page a score based on these.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Prediction : Whether a page's traffic will decline in the next 30 days or so, instead of just finding out whether it's currently trending down.

Label comes from observed future outcome (real traffic measured after the fact, from the daily table) & not a rule applied to the current moment, which is what the starter's weaker proxy does.
We can use any feature from the available list of features to calculate the measured-outcome label, instead of just using the already available features (for eg. trend_direction) to form labels based on a pre-defined rule.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_prev_30d"] > 0].copy()

DECLINE_THRESHOLD = -0.20
df["measured_pct_change"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"]
)
df["measured_decline_label"] = df["measured_pct_change"] <= DECLINE_THRESHOLD

agreement = (df["measured_decline_label"] == (df["trend_direction"] == "down")).mean()
print(f"Agreement between trend_direction rule and prev/last 30d comparison: {agreement*100:.1f}%")
print("-> Near-perfect agreement suggests trend_direction is COMPUTED from these")
print("   same two already-recorded windows — not an independent future observation.")
print("   Both windows already exist in this single snapshot, so this is still a")
print("   same-file comparison, not a genuine prior-window -> later-window forecast")
print("   built around a real decision-point cutoff.")

Agreement between trend_direction rule and prev/last 30d comparison: 99.8%
-> Near-perfect agreement suggests trend_direction is COMPUTED from these
   same two already-recorded windows — not an independent future observation.
   Both windows already exist in this single snapshot, so this is still a
   same-file comparison, not a genuine prior-window -> later-window forecast
   built around a real decision-point cutoff.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The one success metric that we will use for evaluation is 'Precision@K', specifically Precision@50. It gives the % actually declining pages of the top 50 pages flagged as declining.
Another important metric worth considering is % of at-risk traffic caught in the top-K, i.e. out of all the declining pages, how many are caught in the top-K? This is useful because a model can be highly precise in terms of finding out the declining pages in top-50. However, it may not differentiate between a high-traffic & low-traffic page, as it is not trained for it. So, even though a page has low-traffic but is declining, it gets a higher rank in the top 50 pages, than another more important page with high traffic getting declined.

"Good" here is a relative term and not an absolute threshold. If we are using the measured (observed)-outcome label for our analysis then there are 3 possible options for calculating precision@50:
1. Simple base rate : Just counting the no. of declining pages out of the total pages to get a %.
2. Random ranking, averaged over many shuffles : Randomly rank the pages, find their precision@50 and then take an avg. for 1000 such trials.
3. Ranking based on traffic size : pages with higher traffic are ranked first.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_prev_30d"] > 0].copy()

DECLINE_THRESHOLD = -0.20
df["measured_pct_change"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"]
)
df["measured_decline_label"] = (df["measured_pct_change"] <= DECLINE_THRESHOLD).astype(int)

CAPACITY = 50

# STEP 1: the base rate — just count, no ranking at all
base_rate = df["measured_decline_label"].mean()

# STEP 2: prove random ranking converges to that base rate, over many trials
n_trials = 1000
random_precisions = [
    df.sample(frac=1.0, random_state=i).head(CAPACITY)["measured_decline_label"].mean()
    for i in range(n_trials)
]
random_precisions = np.array(random_precisions)

# STEP 3: sort deterministically by prior traffic size
p_traffic = df.sort_values("impressions_prev_30d", ascending=False) \
              .head(CAPACITY)["measured_decline_label"].mean()

print(f"Base rate (no ranking):                 {base_rate:.3f}")
print(f"Random ranking (avg of {n_trials} trials):    {random_precisions.mean():.3f}")
print(f"Sorted by prior traffic size:            {p_traffic:.3f}")

Base rate (no ranking):                 0.613
Random ranking (avg of 1000 trials):    0.613
Sorted by prior traffic size:            0.500


This shows that even the base rate  (obtained through random ranking)is still better than the precision value obtained through traffic-size based ranking. So, we should be using this baseline value of 61.3% as a threshold for our model. The actual precision value obtained from our model should be >61.3% atleast.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Does the row count match the count of unique content_ids?
print(f"Total rows: {len(df):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print(f"Are rows == unique content_ids? {len(df) == df['content_id'].nunique()}")

# One real row, to see what a single unit actually contains
print(df.iloc[0][["content_id", "client_id", "impressions_90d", "trend_direction"]])

# Rule out "one row per client"
pages_per_client = df.groupby("client_id")["content_id"].nunique()
print(f"Pages per client — min: {pages_per_client.min()}, median: {pages_per_client.median():.0f}, "
      f"max: {pages_per_client.max()}")

# Rule out "one row per day" — check for a genuine daily dimension
real_date_cols = [c for c in df.columns if "report_date" in c.lower() or "timestamp" in c.lower()]
print(f"Per-day date columns present: {real_date_cols if real_date_cols else 'NONE'}")

Total rows: 30,000
Unique content_id values: 30,000
Are rows == unique content_ids? True
content_id         content_304f48230142
client_id             client_f369cb89fc
impressions_90d                    3803
trend_direction                    down
Name: 0, dtype: object
Pages per client — min: 3, median: 567, max: 7008
Per-day date columns present: NONE


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.